# 01 — Exploratory Data Analysis (EDA)
## Construction Safety Helmet — YOLO26 (object detection)

Notebook ini **hanya untuk EDA & visualisasi**. Seluruh logika analisis dipanggil
dari modul [`src/eda.py`](../src/eda.py); training **tidak** dilakukan di sini
(training memakai `src/train.py`).

Alur:
1. Load konfigurasi dataset (`dataset/data.yaml`)
2. Jalankan analisis EDA
3. Ringkasan per split
4. Distribusi kelas
5. Statistik bounding box & ukuran gambar
6. Visualisasi utama
7. Interpretasi singkat


## 0. Setup
Menemukan root project secara otomatis & menambahkan `src/` ke `sys.path`, lalu mengimpor `eda`.

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "src" / "eda.py").exists() and (d / "dataset").exists():
            return d
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import eda  # noqa: E402  (dari src/eda.py)

DATA_YAML = PROJECT_ROOT / "dataset" / "data.yaml"
RUNS_EDA = PROJECT_ROOT / "runs" / "eda"
FIG_DIR = RUNS_EDA / "figures"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_YAML    :", DATA_YAML, "(ada)" if DATA_YAML.exists() else "(TIDAK ADA)")


## 1. Load konfigurasi dataset
Membaca `data.yaml`: daftar kelas & jumlah kelas (`nc`).

In [ ]:
data, class_names, nc, root = eda.load_dataset_config(DATA_YAML)
print("dataset root :", root)
print("jumlah kelas :", nc)
print("nama kelas   :", class_names)

splits = eda.resolve_splits(root, data)
print("split terdeteksi:", list(splits.keys()))


## 2. Jalankan analisis EDA
`eda.run_eda(...)` menghitung seluruh statistik, menulis CSV + figur ke
`runs/eda/`, lalu mengembalikan dict hasil. Set `img_sample` (mis. 4000) bila
ingin lebih cepat — statistik ukuran gambar menjadi perkiraan.

In [ ]:
result = eda.run_eda(DATA_YAML, out_dir=RUNS_EDA, img_sample=None, make_plots=True)
print("Total bounding box :", result["total_bboxes"])
print("Output tersimpan di:", RUNS_EDA)


## 3. Ringkasan per split
Jumlah gambar, label, bounding box, gambar tanpa anotasi, dan rata-rata bbox/gambar.

In [ ]:
import pandas as pd
summary_df = pd.read_csv(RUNS_EDA / "eda_summary.csv")
summary_df


## 4. Distribusi kelas
Jumlah bbox per kelas (+ per split) dan persentasenya terhadap total.

In [ ]:
class_df = pd.read_csv(RUNS_EDA / "class_distribution.csv")
class_df


## 5. Statistik bounding box & ukuran gambar
BBox dalam satuan ternormalisasi (relatif terhadap gambar); ukuran gambar dalam pixel.

In [ ]:
bbox_df = pd.read_csv(RUNS_EDA / "bbox_statistics.csv")
img_df = pd.read_csv(RUNS_EDA / "image_statistics.csv")
display(bbox_df)
display(img_df)


## 6. Visualisasi utama
Enam figur disimpan di `runs/eda/figures/` dan ditampilkan di bawah:
1. Jumlah gambar per split
2. Jumlah bbox per kelas
3. Histogram bbox per gambar
4. Scatter width vs height bbox
5. Histogram area bbox
6. Pie proporsi split

In [ ]:
from IPython.display import Image, display

figures = [
    "images_per_split.png",
    "bboxes_per_class.png",
    "bboxes_per_image_hist.png",
    "bbox_wh_scatter.png",
    "bbox_area_hist.png",
    "split_proportion_pie.png",
]
for name in figures:
    path = FIG_DIR / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("[lewati] figur belum ada:", name)


## 7. Interpretasi singkat
Laporan teks lengkap (juga tersimpan di `runs/eda/eda_report.txt`):

In [ ]:
print(result["report_text"])


In [ ]:
# Ringkas poin kunci secara terprogram (selalu sinkron dengan data terbaru)
print("Rasio imbalance kelas (maks/min):", round(result["imbalance_ratio"], 2), "x")
print("Kelas tanpa contoh             :", result["empty_classes"] or "tidak ada")
print(f"Proporsi objek small/medium/large: "
      f"{result['frac_small']*100:.1f}% / {result['frac_medium']*100:.1f}% / {result['frac_large']*100:.1f}%")
print("Dataset cenderung objek kecil  :", result["small_object_dataset"])


### Catatan interpretasi

- **Distribusi kelas:** perhatikan rasio imbalance di atas. Jika kelas minoritas
  (mis. `No-Helmet`) jauh lebih sedikit, pantau metrik **per-kelas** (P/R/mAP)
  saat evaluasi, dan pertimbangkan augmentasi atau penyesuaian bobot.
- **Ukuran objek:** komposisi small/medium/large menunjukkan apakah dataset
  didominasi objek kecil. Jika porsi *small* tinggi, pertahankan `imgsz=640`
  (jangan diturunkan) dan jaga augmentasi yang tidak menghilangkan objek kecil.
- **Gambar tanpa anotasi:** YOLO memperlakukannya sebagai background/negatif —
  pastikan jumlahnya wajar dan memang disengaja.

> **Training tetap dilakukan via script** (`python src/train.py`), bukan di notebook ini.
